# NCA 1D Dataset Visualization

Visualize datasets generated by `data/build_nca1d_dataset.py`.

- Each NCA state is generated as `H x 1`
- Time steps are concatenated along width to form an `H x T` image
- Stored arrays follow the same `PuzzleDataset`-compatible format as the Sudoku builder

In [ ]:
import importlib
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists() and (REPO_ROOT.parent / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import data.build_nca1d_dataset as build_nca1d_dataset
from data.common import PuzzleDatasetMetadata

build_nca1d_dataset = importlib.reload(build_nca1d_dataset)

DATA_DIR = Path(os.environ.get("NCA1D_DATA_DIR", str(REPO_ROOT / "data" / "nca1d-default")))
print("DATA_DIR:", DATA_DIR)


In [ ]:
with open(DATA_DIR / "config.json") as f:
    builder_config = json.load(f)

def load_split(split: str):
    split_dir = DATA_DIR / split
    with open(split_dir / "dataset.json") as f:
        metadata = PuzzleDatasetMetadata(**json.load(f))

    dataset = {
        "inputs": np.load(split_dir / "all__inputs.npy", mmap_mode="r"),
        "labels": np.load(split_dir / "all__labels.npy", mmap_mode="r"),
        "puzzle_identifiers": np.load(split_dir / "all__puzzle_identifiers.npy"),
        "puzzle_indices": np.load(split_dir / "all__puzzle_indices.npy"),
        "group_indices": np.load(split_dir / "all__group_indices.npy"),
        "gzip_ratio": np.load(split_dir / "all__gzip_ratio.npy"),
    }
    return metadata, dataset

train_meta, train_data = load_split("train")
test_meta, test_data = load_split("test")

print("Builder config:")
print(json.dumps(builder_config, indent=2))
print()
print("Train metadata:")
print(train_meta.model_dump())
print()
print("Test metadata:")
print(test_meta.model_dump())


In [ ]:
IMAGE_HEIGHT = int(builder_config["state_height"])
NUM_FRAMES = int(builder_config["num_frames"])
TOKEN_OFFSET = int(builder_config["token_offset"])
NUM_COLORS = int(builder_config["num_colors"])

def decode_image(flat_tokens: np.ndarray) -> np.ndarray:
    return build_nca1d_dataset.unflatten_time_image(
        np.asarray(flat_tokens),
        image_height=IMAGE_HEIGHT,
        num_frames=NUM_FRAMES,
        token_offset=TOKEN_OFFSET,
    )

print("Train arrays:")
for key, value in train_data.items():
    print(f"  {key:18s} shape={value.shape} dtype={value.dtype}")
print()
print("Test arrays:")
for key, value in test_data.items():
    print(f"  {key:18s} shape={value.shape} dtype={value.dtype}")
print()
print("Decoded image shape:", decode_image(train_data["inputs"][0]).shape)
print("Value range after decode:", int(decode_image(train_data["inputs"][0]).min()), int(decode_image(train_data["inputs"][0]).max()))


In [ ]:
def show_examples(dataset, split_name: str, num_examples: int = 6, seed: int = 0):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset["inputs"]), size=min(num_examples, len(dataset["inputs"])), replace=False)
    cmap = plt.get_cmap("tab20", NUM_COLORS)

    fig, axes = plt.subplots(len(indices), 1, figsize=(12, 2.0 * len(indices)), squeeze=False)
    for row_idx, sample_idx in enumerate(indices):
        image = decode_image(dataset["inputs"][sample_idx])
        ax = axes[row_idx, 0]
        ax.imshow(image, cmap=cmap, interpolation="nearest", aspect="auto", vmin=0, vmax=NUM_COLORS - 1)
        ax.set_title(f"{split_name} sample {sample_idx} | gzip={dataset['gzip_ratio'][sample_idx]:.4f}")
        ax.set_ylabel("state")
        ax.set_xlabel("time")
    plt.tight_layout()
    plt.show()

show_examples(train_data, "train", num_examples=6, seed=0)
show_examples(test_data, "test", num_examples=4, seed=1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_data["gzip_ratio"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Train gzip ratio")
axes[0].set_xlabel("compressed / raw")
axes[0].set_ylabel("count")

axes[1].hist(test_data["gzip_ratio"], bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("Test gzip ratio")
axes[1].set_xlabel("compressed / raw")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

train_colors = np.concatenate([decode_image(row).reshape(-1) for row in train_data["inputs"][: min(64, len(train_data['inputs']))]], axis=0)
unique, counts = np.unique(train_colors, return_counts=True)
print("Sampled train color histogram:")
for value, count in zip(unique, counts):
    print(f"  color={int(value):2d} count={int(count):6d}")
